# 06a — Atomic state-dynamics playground

Primo esperimento della nuova famiglia **HayFlow-ESI**. Non addestra il neurone completo: verifica se gli STATE dei meccanismi possono evolvere causalmente di 1 ms meglio della persistenza. Usa soltanto episodi `train`, con ruoli fit/calibration/development disgiunti per seed e snapshot.

In [ ]:
import importlib,json,os,shutil,subprocess,sys,time,zipfile
from pathlib import Path
ELM_REPOSITORY='https://github.com/Zagred47/giada.git';ELM_REF=os.environ.get('HAYFLOW_ELM_REF','main');ROOT=Path('/kaggle/working');ELM_REPO=ROOT/'hayflow_workspace'/'elmneuron';ELM_REPO.parent.mkdir(parents=True,exist_ok=True)
def run(command,cwd=None):print('+',' '.join(map(str,command)),flush=True);subprocess.run(list(map(str,command)),cwd=cwd,check=True)
if not (ELM_REPO/'.git').is_dir():run(['git','clone',ELM_REPOSITORY,ELM_REPO])
run(['git','fetch','origin',ELM_REF],cwd=ELM_REPO);run(['git','checkout','--detach','FETCH_HEAD'],cwd=ELM_REPO);REVISION=subprocess.check_output(['git','rev-parse','HEAD'],cwd=ELM_REPO,text=True).strip();sys.path.insert(0,str(ELM_REPO));importlib.invalidate_caches();print({'revision':REVISION})

## 1. Autorità sperimentale e dataset

Il notebook richiede il NO-GO esatto di 05t, il dataset targeted base e il top-up contenente il manifest composito. Caricare i tre Dataset Kaggle come Input.

In [ ]:
from src.hayflow_model.rollout_aware_architecture_canary import discover_indexed_artifact_source
from src.hayflow_model.atomic_state_dynamics_playground import EXPECTED_05T_INDEX_SHA256
INPUT_ROOT=Path('/kaggle/input')
artifact_override=os.environ.get('HAYFLOW_05T_ARTIFACT');ARTIFACT_05T_SOURCE=discover_indexed_artifact_source(INPUT_ROOT,EXPECTED_05T_INDEX_SHA256,override=Path(artifact_override) if artifact_override else None);assert ARTIFACT_05T_SOURCE is not None,'Artefatto 05t esatto non trovato: aggiungi hayflow_consolidated_autoregressive_go_no_go agli Input Kaggle.'
def extract_zip_safely(source,destination):
 source,destination=Path(source),Path(destination);marker=destination/'.source_stamp';stamp=f'{source.stat().st_size}:{source.stat().st_mtime_ns}'
 if marker.is_file() and marker.read_text().strip()==stamp:return destination
 if destination.exists():shutil.rmtree(destination)
 destination.mkdir(parents=True);root=destination.resolve()
 with zipfile.ZipFile(source) as archive:
  for member in archive.infolist():
   target=(destination/member.filename).resolve();assert target==root or root in target.parents,member.filename
  archive.extractall(destination)
 marker.write_text(stamp);return destination
topup_override=os.environ.get('HAYFLOW_TOPUP_V3');topup_candidates=([Path(topup_override).expanduser()] if topup_override else [])+list(INPUT_ROOT.rglob('hayflow_bap_validation_support_topup_v3.zip'))+[p.parent for p in INPUT_ROOT.rglob('composite_dataset_manifest.json')]
TOPUP_SOURCE=next((p.resolve() for p in topup_candidates if p.exists()),None);assert TOPUP_SOURCE is not None,'Top-up BAP v3 non trovato.';TOPUP_ROOT=extract_zip_safely(TOPUP_SOURCE,'/kaggle/working/hayflow06a_topup') if TOPUP_SOURCE.is_file() else TOPUP_SOURCE
manifest_candidates=list(Path(TOPUP_ROOT).rglob('composite_dataset_manifest.json'));assert len(manifest_candidates)==1,manifest_candidates;COMPOSITE_MANIFEST=manifest_candidates[0]
base_override=os.environ.get('HAYFLOW_BASE_DATASET');base_candidates=([Path(base_override).expanduser()] if base_override else [])+[p.parent for p in INPUT_ROOT.rglob('transition_dataset.h5') if 'targeted' in str(p).lower() and 'topup' not in str(p).lower()]+[p for p in INPUT_ROOT.rglob('archive.zip') if 'hayflow-targeted-transition-dataset' in str(p).lower()]
BASE_SOURCE=next((p.resolve() for p in base_candidates if p.exists()),None);assert BASE_SOURCE is not None,'Dataset base targeted v1.1 non trovato.';print({'05t':str(ARTIFACT_05T_SOURCE),'base':str(BASE_SOURCE),'composite_manifest':str(COMPOSITE_MANIFEST)})

In [ ]:
from src.hayflow_data import prepare_composite_flowmap_bundle
hash_started={};hash_last={}
def hash_progress(name,done,total):
 now=time.monotonic();hash_started.setdefault(name,now);percent=int(100*done/total)
 if percent>=hash_last.get(name,-10)+10 or done==total:
  elapsed=now-hash_started[name];rate=done/max(elapsed,1e-9);eta=(total-done)/max(rate,1e-9);print(f'[HayFlow 06a][SHA-256 {name}] {percent}% ETA {eta/60:.1f} min',flush=True);hash_last[name]=percent
bundle=prepare_composite_flowmap_bundle(COMPOSITE_MANIFEST,base_source=BASE_SOURCE,progress=hash_progress);assert bundle.manifest['valid'] and bundle.transition_count==29880 and not bundle.manifest['physical_merge_performed'];print({'dataset_valid':True,'transitions':bundle.transition_count,'fingerprint':bundle.fingerprint})

## 2. Preflight train-only

Costruisce ruoli disgiunti, indicizza gli STATE dei meccanismi e fitta le trasformazioni esclusivamente su `train/fit`. L’output mostrato è intenzionalmente compatto.

In [ ]:
import yaml
from IPython.display import display
from src.hayflow_model import AtomicStateDynamicsConfig,AtomicStateDynamicsPlayground
cfg=yaml.safe_load((ELM_REPO/'configs/hayflow/hayflow_atomic_state_dynamics_playground.yml').read_text());config=AtomicStateDynamicsConfig.from_mapping(cfg['atomic_state_dynamics_playground'])
OUTPUT_DIR=Path('/kaggle/working/artifacts/hayflow_atomic_state_dynamics_playground');assert not OUTPUT_DIR.exists(),f'Output già presente: {OUTPUT_DIR}. Avvia una sessione nuova.'
session=AtomicStateDynamicsPlayground(bundle,OUTPUT_DIR,config,ARTIFACT_05T_SOURCE,code_revision=REVISION);preflight=session.prepare_playground();display({'valid':preflight['valid'],'architecture':preflight['architecture_family'],'roles':preflight['role_transition_counts'],'mechanism_coordinates':preflight['mechanism_state_coordinate_count'],'semantic_groups':preflight['semantic_group_count'],'state_splits_read':preflight['state_and_outcome_splits_read'],'validation_state_accessed':preflight['validation_state_accessed']});assert preflight['valid'] and preflight['state_and_outcome_splits_read']==['train'] and not preflight['validation_state_accessed'] and not preflight['test_state_accessed']

## 3. Pilot appaiato e rollout dello stato

I due bracci hanno capacità e inizializzazione identiche. Il secondo usa `V_t+1−V_t` soltanto come controfattuale diagnostico. Il tracker stampa una riga ogni 25 step, senza array o stati completi.

In [ ]:
try:
 pilot_report=session.run_one_step_pilot();rollout_report=session.evaluate_teacher_voltage_rollouts();final_report=session.finalize(pilot_report,rollout_report)
finally:
 session.close()
one_step={arm:{'rmse':round(row['development']['normalized_delta_rmse'],4),'persistence':round(row['development']['persistence_normalized_delta_rmse'],4),'gain':round(row['development']['improvement_vs_persistence_fraction'],4),'active_gain':round(row['development']['active_improvement_vs_persistence_fraction'],4)} for arm,row in pilot_report['runs'].items()}
horizons={arm:{key:round(value['improvement_vs_persistence_fraction'],4) for key,value in rows.items()} for arm,rows in rollout_report['arms'].items()}
display({'valid':final_report['valid'],'decision_grade':final_report['decision_grade'],'diagnosis':final_report['diagnosis'],'one_step':one_step,'recursive_state_gain_vs_persistence':horizons,'validation_state_accessed':final_report['validation_state_accessed'],'next_step':final_report['next_step']});assert final_report['valid'] and not final_report['decision_grade'] and not final_report['candidate_selection_performed'] and not final_report['validation_state_accessed'] and not final_report['test_state_accessed']

In [ ]:
from IPython.display import Image,display
display(Image(filename=str(OUTPUT_DIR/'figures/atomic_state_pilot_summary.png')))

## 4. Crea e scarica lo ZIP

La cella usa il downloader browser stabile del progetto: ZIP in `/kaggle/working`, base64, `Blob` e click temporaneo.

In [ ]:
from shutil import make_archive
import base64
from IPython.display import Javascript,display
zip_path=Path(make_archive('/kaggle/working/hayflow_atomic_state_dynamics_playground','zip',root_dir=OUTPUT_DIR.parent,base_dir=OUTPUT_DIR.name));payload=base64.b64encode(zip_path.read_bytes()).decode('ascii');filename=zip_path.name
display(Javascript(f"""const binary=atob('{payload}');const bytes=new Uint8Array(binary.length);for(let i=0;i<binary.length;i++)bytes[i]=binary.charCodeAt(i);const blob=new Blob([bytes],{{type:'application/zip'}});const url=URL.createObjectURL(blob);const a=document.createElement('a');a.href=url;a.download='{filename}';document.body.appendChild(a);a.click();a.remove();setTimeout(()=>URL.revokeObjectURL(url),60000);"""));print({'zip':str(zip_path),'size_mib':round(zip_path.stat().st_size/2**20,2),'download':'avviato dal browser'})